# Document Search with LangChain

This example shows how to use the Python [LangChain](https://python.langchain.com/docs/get_started/introduction) library to run a text-generation request on open-source LLMs and embedding models using the OpenAI SDK, then augment that request using the text stored in a collection of local PDF documents.

### <u>Requirements</u>
1. As you will accessing the LLMs and embedding models through Vector AI Engineering's Kaleidoscope Service (Vector Inference + Autoscaling), you will need to request a KScope API Key:

   Run the following command (replace ```<user_id>``` and ```<password>```) from **within the cluster** to obtain the API Key. The ```access_token``` in the output is your KScope API Key.
  ```bash
  curl -X POST -d "grant_type=password" -d "username=<user_id>" -d "password=<password>" https://kscope.vectorinstitute.ai/token
  ```
2. After obtaining the `.env` configurations, make sure to create the ```.kscope.env``` file in your home directory (```/h/<user_id>```) and set the following env variables:
- For local models through Kaleidoscope (KScope):
    ```bash
    export OPENAI_BASE_URL="https://kscope.vectorinstitute.ai/v1"
    export OPENAI_API_KEY=<kscope_api_key>
    ```
- For OpenAI models:
   ```bash
   export OPENAI_BASE_URL="https://api.openai.com/v1"
   export OPENAI_API_KEY=<openai_api_key>
   ```
3. (Optional) Upload some pdf files into the `source_documents` subfolder under this notebook. We have already provided some sample pdfs, but feel free to replace these with your own.

In [ ]:
import requests
import getpass

url = "https://kscope.vectorinstitute.ai/token"
username = 'ws_mshaykat'
password = getpass.getpass("Enter your password: ")

data = {
    'grant_type': 'password',
    'username': username,
    'password': password
}

response = requests.post(url, data=data)
if response.status_code == 200:
    access_token = response.json()['access_token']
    print("Access Token:", access_token)
else:
    print("Error:", response.status_code, response.text)

In [ ]:
OPENAI_BASE_URL="https://kscope.vectorinstitute.ai/v1"
OPENAI_API_KEY=access_token
DOC_PATH = "/projects/RAG2/scotia-2/Datasets-Scotia-2/"
DATASET_NAMES = ['Agriculture_txt', 'Transport_txt', 'IBIS', 'Auto_txt']

## Set up the RAG workflow environment

#### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import requests
import sys

from pathlib import Path

from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter

#### Load config files

In [3]:
# Add root folder of the rag_bootcamp repo to PYTHONPATH
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))

from utils.load_secrets import load_env_file
load_env_file()

In [4]:
GENERATOR_BASE_URL = OPENAI_BASE_URL # os.environ.get("OPENAI_BASE_URL")
OPENAI_API_KEY = OPENAI_API_KEY # os.environ.get("OPENAI_API_KEY")

#### Set up some helper functions

In [5]:
import re
import pandas as pd
import nltk

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(\w)- (\w)', r'\1\2', text)
    return text.strip()

def pretty_print_docs(docs):
    # docs = clean_text(docs)
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + clean_text(d.page_content) for i, d in enumerate(docs)]
        )
    )

#### Make sure other necessary items are in place

##### Nevigate the Datasets

In [ ]:
for DATASET_NAME in DATASET_NAMES:
    DATASET_DIR_PATH = DOC_PATH + DATASET_NAME
    file_list = os.listdir(DATASET_DIR_PATH)
    print(DATASET_DIR_PATH)
    print(file_list)
    print("******************************************************************")

In [ ]:
def format_size(size):
    for unit in ['bytes', 'KB', 'MB', 'GB']:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} TB"

DATASET_DIR_PATH = DOC_PATH + DATASET_NAMES[2]
file_list = os.listdir(DATASET_DIR_PATH)
for file_name in file_list:
    full_path = os.path.join(DATASET_DIR_PATH, file_name)
    if os.path.isfile(full_path):
        file_size = os.path.getsize(full_path)
        print(f"{file_name}: {format_size(file_size)}")

In [6]:
# Look for the source_documents folder and make sure there is at least 1 pdf file here
contains_pdf = False
directory_path = DOC_PATH + DATASET_NAMES[2]
if not os.path.exists(directory_path):
    print(f"ERROR: The {directory_path} subfolder must exist under this notebook")
for filename in os.listdir(directory_path):
    contains_pdf = True if ".pdf" in filename else contains_pdf
if not contains_pdf:
    print(f"ERROR: The {directory_path} subfolder must contain at least one .pdf file")

#### Choose LLM and embedding model

In [7]:
GENERATOR_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

## Start with a basic generation request without RAG augmentation

Let's start by asking Llama-3.1 a difficult, domain-specific question we don't expect it to have an answer to. A simple question like "*What is the capital of France?*" is not a good question here, because that's world knowledge that we expect the LLM to know.

Instead, we want to ask it a question that is domain-specific and it won't know the answer to. A good example would be an obscure detail buried deep within a company's annual report. For example:

*How many Vector scholarships in AI were awarded in 2022?*

In [8]:
query = "How many Vector scholarships in AI were awarded in 2022?"

## Now send the query to the open source model using KScope

In [9]:
llm = ChatOpenAI(
    model=GENERATOR_MODEL_NAME,
    temperature=0,
    max_tokens=None,
    base_url=GENERATOR_BASE_URL,
    api_key=OPENAI_API_KEY
)
message = [
    ("human", query),
]
try:
    result = llm.invoke(message)
    print(f"Result: \n\n{result.content}")
except Exception as err:
    if "Error code: 503" in err.message:
        print(f"The model {GENERATOR_MODEL_NAME} is not ready yet.")
    else:
        raise

Result: 

I don't have access to real-time data or specific information about the number of Vector scholarships in AI awarded in 2022. For the most accurate and up-to-date information, I recommend checking directly with Vector Institute or their official website. They would have the most current details on their scholarship programs and awards.


Without additional information, Llama-3.1 is unable to answer the question correctly. **Vector in fact awarded 109 AI scholarships in 2022.** Fortunately, we do have that information available in Vector's 2021-22 Annual Report, which is available in the `source_documents` folder. Let's see how we can use RAG to augment our question with a document search and get the correct answer.

## Ingestion: Load and store the documents from `source_documents`

Start by reading in all the PDF files from `source_documents`, break them up into smaller digestible chunks, then encode them as vector embeddings.

#### Load IBIS PDF Data

In [ ]:
# Load the pdfs
directory_path = DOC_PATH + DATASET_NAMES[2]
loader = PyPDFDirectoryLoader(directory_path)
docs = loader.load()
print(f"Number of source documents: {len(docs)}")

# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=32)
pdf_doc_chunks = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(pdf_doc_chunks)}")

In [10]:
pdf_doc_chunks[0]

Number of source documents: 42
Number of text chunks: 228


In [ ]:
docs[0]

#### Load Auto Text Data of US & CA

In [ ]:
def read_csv_from_directory(directory_path):
    dataframes= []
    for filename in os.listdir(directory_path):
        if filename.endswith('.csv'): 
            file_path = os.path.join(directory_path, filename)
            df = pd.read_csv(file_path)
            df["source"] = filename
            # print(df.head(1))
            dataframes.append(df)
    return pd.concat(dataframes, ignore_index=True)

In [ ]:
directory_path = DOC_PATH + DATASET_NAMES[3]
data = read_csv_from_directory(directory_path)

text_data = data['story'].astype(str).tolist()  # Adjust column name
cleaned_text_list = []
for text in text_data:
    cleaned_text_list.append(clean_text(text))

    
# auto_data_dic = data.to_dict(orient='records')
# auto_data_dic[1]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,  # Maximum chunk size
    chunk_overlap=32  # Overlap between chunks; adjust as necessary
)

text_docs = text_splitter.create_documents(cleaned_text_list)
print(text_docs[0])

csv_doc_chunks = text_splitter.split_documents(text_docs)
print(f"Number of text chunks: {len(csv_doc_chunks)}")

#### Define the embeddings model

In [11]:
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

print(f"Setting up the embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Setting up the embeddings model...


## Retrieval: Make the document chunks available via a retriever

The retriever will identify the document chunks that most closely match our original query. (This takes about 1-2 minutes)

In [12]:
vectorstore = FAISS.from_documents(csv_doc_chunks, embeddings)
# vectorstore = FAISS.from_documents(pdf_doc_chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Retrieve the most relevant context from the vector store based on the query
retrieved_docs = retriever.invoke(query)

Let's see what results it found. Important to note, these results are in the order the retriever thought were the best matches.

In [13]:
pretty_print_docs(retrieved_docs)

Document 1:

5 
Annual Report 2021–22 Vector Institute
SPOTLIGHT ON FIVE YEARS OF AI 
LEADERSHIP FOR CANADIANS 
SINCE THE VECTOR INSTITUTE WAS FOUNDED IN 2017: 
2,080+ 
Students have graduated from 
Vector-recognized AI programs and 
study paths $6.2 M 
Scholarship funds committed to 
students in AI programs 3,700+ 
Postings for AI-focused jobs and 
internships ofered on Vector’s 
Digital Talent Hub $103 M 
In research funding committed to 
Vector-afliated researchers 
94 
Research awards earned by
----------------------------------------------------------------------------------------------------
Document 2:

26 
  VECTOR SCHOLARSHIPS IN 
AI ATTRACT TOP TALENT TO ONTARIO UNIVERSITIES 
109 
Vector Scholarships in AI awarded 
34 
Programs 
13 
Universities 
351 
Scholarships awarded since the 
program launched in 2018 Supported with funding from the Province of Ontario, the Vector Institute Scholarship in Artifcial Intelligence (VSAI) helps Ontario universities to attract the best and b

## Now send the query to the RAG pipeline

In [14]:
rag_pipeline = RetrievalQA.from_llm(llm=llm, retriever=retriever)
result = rag_pipeline.invoke(input=query)
print(f"Result: \n\n{result['result']}")

Result: 

The text does not provide the number of Vector Scholarships in AI awarded in 2022. It does provide the total number of Vector Scholarships in AI awarded since the program launched in 2018, which is 109.


The model provides the correct answer (109) using the retrieved information.

But it also continues with the following 2 scenarios (as of now) due to stochasticity:
1. It sometimes outputs another sentence which seems to be hallucinated as it gets confused between the total scholarships (351) and those awarded just in 2022.
2. It sometimes just says *I don't know* because its not sure about the year.

In [2]:
import re
import nltk
import io
import ssl
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
ssl._create_default_https_context = ssl._create_unverified_context

# Download the words resource (if you haven't done so already)
nltk.download('words', quiet=True)
nltk.download('wordnet', quiet=True)


# Load the words data from NLTK directly into memory
from nltk.corpus import words

# Get the list of words
word_list = words.words()

# Convert to a set for better performance on membership checks
word_set = set(word_list)

# Example usage
print(f"Number of words: {len(word_set)}")
print("Sample words:", list(word_set)[:10])  # Print first 10 words


Number of words: 235892
Sample words: ['Amerind', 'spreeuw', 'Algarsife', 'apochromatism', 'columellate', 'latro', 'portentous', 'Manitoban', 'duodecahedron', 'anteriority']


In [3]:
new_word_list = []
for w in word_list:
    if len(w) > 1:
        new_word_list.append(w)
word_set = set(new_word_list)
for w in word_set:
    if len(w) <= 1:
        print(w)

In [7]:
special_characters = [
    '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', 
    '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', 
    '^', '_', '`', '{', '|', '}', '~', ' ', '\t', '\n'
]

def ends_with_special_character(word):
    # print(list(word.strip()).pop(-1))
    if list(word).pop(-1) in special_characters:
        return True
    else:
        return False
    # return bool(re.search(r'[!\"#$%&\'()*+,\-./:;<=>?@[\\\]^_`{|}~]$', word))

def get_base_word_ing(word):
    lemmatizer = WordNetLemmatizer()
    base_word = lemmatizer.lemmatize(word, pos='v')
    return base_word

def is_correct_word(word):
    return bool(wn.synsets(word))


def reconstruct_broken_words(text):
    # Split the text into parts (words)
    text = " ".join(text)
    word_parts = text.split(" ")
    print(word_parts)
    print("")
    print("")
    print("")
    
    reconstructed = []
    current_word = ""

    part_queue = []
    while word_parts:
        # Check if the current part can be a word
        temp_part = word_parts.pop(0).strip()
        # print(temp_part)
        if temp_part in special_characters:
            reconstructed.append(temp_part)
            # print(temp_part)
        else:
            # print(temp_part)
            part_queue.append(temp_part.strip())
            # print(part_queue)
            current_word = "".join(part_queue)
            # print(current_word)
            s_word = current_word.lower()    
            if ends_with_special_character(s_word):
                s_word = s_word[:len(s_word)-1]
            if s_word.endswith('s') and len(current_word) > 2:
                if s_word.endswith('ies'):
                    s_word = s_word[:len(s_word)-3]
                if s_word.endswith('es'):
                    s_word = s_word[:len(s_word)-2]
                else:
                    s_word = s_word[:len(s_word)-1] 
            if s_word.endswith('ing'):
                s_word = get_base_word_ing(s_word)
            
            if s_word in word_set:
                reconstructed.append(current_word)
                # Start a new current word
                current_word = ""
                s_word = ""
                part_queue = []
            else:   
                if is_correct_word(s_word):
                    reconstructed.append(current_word)
                
                    # Start a new current word
                    current_word = ""
                    s_word = ""
                    part_queue = []
        print(reconstructed)
    if current_word:
        reconstructed.append(current_word)
    return ' '.join(reconstructed)

# Input broken text
broken_text = [
    "Technol ogical adv ancemen ts such as pr ecision agricul ture are beginning t "
    "o reshape f arming pr actices, impr oving e fficienc y and sustainability . "
]
# broken_text = [
#     "Technol ogical adv ancemen ts such as pr ecision agricul ture are beginning t "
#     "o reshape f arming pr actices, impr oving e fficienc y and sustainability . "
#     "Overall, indus try r evenue has climbed a t a C AGR o f 0.9% t o reach an e "
#     "xpec ted $2.85 bil lion in 202 4 after a decr ease of 10.3% in the y ear. "
#     "Over the pas t five years, corn pric es e xperienc ed no table increases, "
#     "especial ly be tween 2021 and 2023, driv en b y global demand and high c "
#     "osts for input s such as f ertiliz er and oil. In"
# ]

# Clean and reconstruct the broken text
cleaned_text = reconstruct_broken_words(broken_text)

# Output the cleaned and reconstructed text
print(cleaned_text)

['Technol', 'ogical', 'adv', 'ancemen', 'ts', 'such', 'as', 'pr', 'ecision', 'agricul', 'ture', 'are', 'beginning', 't', 'o', 'reshape', 'f', 'arming', 'pr', 'actices,', 'impr', 'oving', 'e', 'fficienc', 'y', 'and', 'sustainability', '.', '']



[]
['Technological']
['Technological']
['Technological']
['Technological', 'advancements']
['Technological', 'advancements', 'such']
['Technological', 'advancements', 'such', 'as']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements', 'such', 'as', 'pr']
['Technological', 'advancements',

In [43]:
if "precision" in word_set:
    print("advancements")

advancements


In [88]:
import nltk
from nltk.corpus import wordnet as wn

# Download WordNet if you haven't already
nltk.download('wordnet', quiet=True)

def is_correct_word(word):
    return bool(wn.synsets(word))

# Check if "sustainability" is a correct word
word_to_check = 'pr'
if is_correct_word(word_to_check):
    print(f'"{word_to_check}" is a valid word.')
else:
    print(f'"{word_to_check}" is not a valid word.')


"pr" is a valid word.


In [8]:
import pdfplumber
import os

def load_pdfs_from_directory(directory_path):
    documents = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith('.pdf'):
            file_path = os.path.join(directory_path, filename)
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()  # Extract text from each page
                    documents.append({'filename': filename, 'text': text})
    
    return documents

# Example usage
pdf_directory_path = '/Users/shaykatmdabdulmutalab/Documents/RAG_BOOTCAMP/metarials'  # Replace with your directory path
pdf_documents = load_pdfs_from_directory(pdf_directory_path)

for doc in pdf_documents:
    print(f"Document: {doc['filename']}\nText: {doc['text']}\n")


Document: RAG2 - Build Days Schedule.pdf
Text: From To Day 1: Jan 28 Day 2: Jan 29 Day 3: Jan 30
9:30 9:40
9:40 9:50 Hands-on Engage in breakout sessions to develop your proof of concept
9:50 10:00 Hands-on and compile an end-of-day report. Facilitators are available via
Hands-on Hands-on
10:00 10:10 Slack or can join your breakout room for urgent support.
10:10 10:20
10:20 10:30
Scheduled Facilitation Next Frontier Join complementary talks offering fresh perspectives and
10:30 10:40 Talks future possibilities in RAG.
10:40 10:50
10:50 11:00
Scheduled Facilitation Scheduled Facilitation
11:00 11:10 Querying a network of knowledge Team At the end of the day, teams will present their progress and
11:10 11:20 with llama-index-networks Knowledge key learnings to foster collaboration and share insights with
11:20 11:30 Val Andrei Fajardo - Applied ML Sharing other teams.
11:30 11:40 Talk: Biodiversity RAG System Scientist - Vector Institute
11:40 11:50 Nate Lesperance - Research Hands-on Ha